In [121]:
import ast
import glob
from collections import defaultdict
from datetime import date

import wandb
import numpy as np
import pandas as pd

# Set display options to show all columns and format floats nicely
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

user = "imos-buildingenergy-1"


today = date.today()
api = wandb.Api()

# # Find all csv files in the current directory
csv_files = glob.glob("*.csv")
# # Collect all the names of the csv files without the extension
csv_names = [csv_file[:-4] for csv_file in csv_files]
project_name = "TabPHM-autoencoders-checks_02_10_night"
 

if project_name not in csv_names:
    runs = api.runs(f"{user}/{project_name}")

    summary_list, config_list, name_list = [], [], []
    for run in runs:
        # .summary contains the output keys/values for metrics like accuracy.
        #  We call ._json_dict to omit large files
        summary_list.append(run.summary._json_dict)

        # .config contains the hyperparameters.
        #  We remove special values that start with _.
        config_list.append(
            {k: v for k, v in run.config.items() if not k.startswith("_")}
        )

        # .name is the human-readable name of the run.
        name_list.append(run.name)

    runs_df = pd.DataFrame(
        {"summary": summary_list, "config": config_list, "name": name_list}
    )

    runs_df.to_csv(f"{project_name}.csv")
else:
    runs_df = pd.read_csv(f"{project_name}.csv", index_col=0)

    for row in runs_df.iloc:
        row["summary"] = ast.literal_eval(row["summary"])
        row["config"] = ast.literal_eval(row["config"])


for row in runs_df.iloc:
    row["summary"].update(row["config"])

lst = [i["summary"] for i in runs_df.iloc]
df = pd.DataFrame.from_dict(lst)

df_init = df.copy()

# Get average epoch run time
df["epoch_run_time"] = df["_runtime"] / df["epoch"]



### Go over different config groups and flatten the content from the nested dictionaries into separate columns

In [122]:
def normalize_column(df, column_to_normalize):
    # Use json_normalize to flatten the nested dictionaries into separate columns
    flattened_df = pd.json_normalize(df[column_to_normalize])
    # Rename columns to include 'nested_column' prefix
    flattened_df.columns = [
        f"{column_to_normalize}.{col}" for col in flattened_df.columns
    ]
    # Concatenate the flattened DataFrame with the original DataFrame
    result_df = pd.concat([df, flattened_df], axis=1)
    # Get new columns names
    new_columns = flattened_df.columns
    # Drop the original nested column if needed
    result_df.drop(column_to_normalize, axis=1, inplace=True)
    return result_df, new_columns


# Config columns to normalize
columns_to_normalize = ["optimizer", "datasource", "task_definition","model", "dataset", "callbacks", "paths"]
droped_columns = []
# Keep track of config columns added
config_columns = []
for column in columns_to_normalize:
    df, columns = normalize_column(df, column)
    config_columns.extend(columns)

In [123]:


# Describe only the numerical columns
print("--- 📊 Numerical Columns Summary ---")
display(df.describe(include='number'))

# Describe only the object/categorical columns
print("\n--- 📝 Object/Categorical Columns Summary ---")
display(df.describe(include=['object']))

# check columns for nans and provide statistics
nan_stats = df.isna().sum()
nan_stats = nan_stats[nan_stats > 0]
print(f"\n--- ⚠️ Columns with NaN Values, {(nan_stats.shape[0]/ len(df.columns)) *100}% has NaN values ---")

# for every column calucalate also the percentage of nans
nan_stats = pd.DataFrame(nan_stats, columns=["num_nans"])
nan_stats["percentage_nans"] = (nan_stats["num_nans"] / len(df)) * 100
nan_stats = nan_stats.sort_values(by="percentage_nans", ascending=False)
display(nan_stats)


# Take all the columns that all more the 90% nans
nan_cols_to_drop = nan_stats[nan_stats["percentage_nans"] > 90].index.tolist()
print(f"Dropping {len(nan_cols_to_drop)} columns with more than 90% NaNs")
# Remove columns that are not needed (we shouldn't vary them or their variation is not interesting)
df = df.drop(nan_cols_to_drop, axis=1)
droped_columns.extend(nan_cols_to_drop)

--- 📊 Numerical Columns Summary ---


/home/levtel/miniconda3/envs/phm2/lib/python3.12/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,_runtime,_step,_timestamp,epoch,lr-Adam,train/loss_step,trainer/global_step,seed,num_workers,val_batch_size,prefetch_factor,test_batch_size,train_batch_size,model/params/total,model/params/trainable,AvgTime/train_batch_mean,AvgTime/train_batch_std,AvgTime/train_epoch_mean,AvgTime/train_epoch_std,AvgTime/val_batch_mean,AvgTime/val_batch_std,AvgTime/val_epoch_mean,AvgTime/val_epoch_std,test/loss,test/mae,test/mse,test/rmse,train/loss_epoch,train/mae,train/mse,train/rmse,val/loss,val/mae,val/mse,val/rmse,epoch_run_time,optimizer.scheduler.scheduler_params.gamma,optimizer.scheduler.scheduler_params.step_size,optimizer.parameters.lr,optimizer.parameters.weight_decay,datasource.fold,task_definition.stride,task_definition.seq_len,task_definition.pred_len,task_definition.label_len,model.latent_dim,model.dropout_prob,model.input_seq_len,model.input_channels,dataset.dataset_cfg.stride,dataset.dataset_cfg.seq_len,dataset.dataset_cfg.pred_len,dataset.dataset_cfg.label_len,callbacks.model_summary.max_depth,callbacks.early_stopping.patience,callbacks.early_stopping.min_delta,callbacks.model_checkpoint.save_top_k
count,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,54.00,54.00,54.00,54.00,54.00,2.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00
mean,2048.47,2062.31,1759385863.91,18.07,0.00,2286.12,98532.67,42.00,4.00,986.67,10.00,986.67,433.04,755587.56,755587.56,0.02,0.00,84.50,3.52,0.01,0.00,2.95,0.11,1478.75,4.90,4436.25,6.56,1283402.15,16.28,3845266.82,245.63,1420.70,4.79,4261.00,6.46,inf,0.50,14.44,0.00,0.00,0.00,1.00,64.00,0.00,0.00,124.44,0.20,64.00,2.96,1.00,64.00,0.00,0.00,-1.00,25.00,0.00,1.00
std,2421.98,2536.71,14665.26,5.64,0.00,16324.21,127058.57,0.00,0.00,192.15,0.00,192.15,353.43,401473.83,401473.83,0.00,0.00,53.32,1.62,0.00,0.00,0.31,0.06,10134.90,29.49,30404.70,38.29,6388875.42,69.58,19167433.89,1117.15,9930.07,28.89,29790.36,37.51,NaN,0.00,5.02,0.00,0.00,0.00,0.00,0.00,0.00,0.00,18.30,0.00,0.00,0.19,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
min,20.00,21.00,1759357555.33,0.00,0.00,0.00,1049.00,42.00,4.00,16.00,10.00,16.00,16.00,183208.00,183208.00,0.02,0.00,28.31,0.74,0.01,0.00,2.46,0.03,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,30.20,0.50,10.00,0.00,0.00,0.00,1.00,64.00,0.00,0.00,32.00,0.20,64.00,2.00,1.00,64.00,0.00,0.00,-1.00,25.00,0.00,1.00
25%,699.50,544.00,1759375940.58,20.00,0.00,0.00,22120.00,42.00,4.00,1024.00,10.00,1024.00,128.00,380172.00,380172.00,0.02,0.00,41.32,2.49,0.01,0.00,2.69,0.05,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,51.41,0.50,10.00,0.00,0.00,0.00,1.00,64.00,0.00,0.00,128.00,0.20,64.00,3.00,1.00,64.00,0.00,0.00,-1.00,25.00,0.00,1.00
50%,1103.00,986.00,1759389995.93,20.00,0.00,0.00,44220.00,42.00,4.00,1024.00,10.00,1024.00,256.00,380442.00,380442.00,0.02,0.00,69.41,3.08,0.01,0.00,2.98,0.10,0.00,0.02,0.01,0.04,0.00,0.02,0.01,0.05,0.00,0.02,0.01,0.04,92.72,0.50,10.00,0.00,0.00,0.00,1.00,64.00,0.00,0.00,128.00,0.20,64.00,3.00,1.00,64.00,0.00,0.00,-1.00,25.00,0.00,1.00
75%,2998.00,3196.75,1759397213.50,20.00,0.01,0.07,154770.00,42.00,4.00,1024.00,10.00,1024.00,512.00,1167436.00,1167436.00,0.02,0.00,109.32,4.52,0.01,0.00,3.23,0.14,0.02,0.10,0.06,0.14,0.76,0.44,2.28,0.87,0.03,0.12,0.09,0.17,170.14,0.50,20.00,0.01,0.00,0.00,1.00,64.00,0.00,0.00,128.00,0.20,64.00,3.00,1.00,64.00,0.00,0.00,-1.00,25.00,0.00,1.00
max,12956.98,13888.00,1759414024.34,20.00,0.01,119997.78,692399.00,42.00,4.00,1024.00,10.00,1024.00,1024.00,1167436.00,1167436.00,0.02,0.01,172.91,7.29,0.01,0.01,3.48,0.27,70230.30,204.25,210690.83,265.01,36766636.00,359.69,110299888.00,6063.55,70230.30,204.25,210690.83,265.01,inf,0.50,20.00,0.01,0.00,0.00,1.00,64.00,0.00,0.00,128.00,0.20,64.00,3.00,1.00,64.00,0.00,0.00,-1.00,25.00,0.00,1.00



--- 📝 Object/Categorical Columns Summary ---


,_wandb,loss,tags,cache,logger,trainer,exp_name,ckpt_path,evaluator,hydra_cfg,datamodule,transforms,profiler_cfg,subset_range,experiment_group,plots/test/reconstructions_overview,plots/train/reconstructions_overview,plots/val/reconstructions_overview,optimizer._target_,optimizer.optimizer_id,optimizer.scheduler.scheduler_id,datasource._target_,datasource.data_dir,datasource.data_name,datasource.task_mode,datasource.cache_dir,task_definition.task_type,task_definition.model.data_requirements.input_tensors,model.kernels,model.strides,model._target_,model.dilations,model.output_channels,model.blocks_config,dataset._target_,dataset.dataset_cfg._target_,callbacks.model_timer._target_,callbacks.model_summary._target_,callbacks.early_stopping.mode,callbacks.early_stopping.monitor,callbacks.early_stopping._target_,callbacks.early_stopping.stopping_threshold,callbacks.early_stopping.divergence_threshold,callbacks.early_stopping.check_on_train_epoch_end,callbacks.model_checkpoint.mode,callbacks.model_checkpoint.dirpath,callbacks.model_checkpoint.monitor,callbacks.model_checkpoint._target_,callbacks.model_checkpoint.filename,callbacks.model_checkpoint.every_n_epochs,callbacks.model_checkpoint.every_n_train_steps,callbacks.model_checkpoint.train_time_interval,callbacks.model_checkpoint.save_on_train_epoch_end,callbacks.learning_rate_monitor._target_,callbacks.learning_rate_monitor.logging_interval,paths.log_dir,paths.ckpt_dir,paths.data_dir,paths.plot_dir,paths.root_dir,paths.work_dir,paths.cache_path,paths.output_dir,paths.eval_details,paths.artifacts_dir,paths.model_workdir,paths.model_cache_dir
count,54,54,54,54,54,54,54,0,54,54,54,54,0,0,54,48,50,50,54,54,54,54,52,54,54,2,54,54,54,54,54,54,54,2,54,54,54,54,54,54,54,0,0,0,54,54,54,54,54,0,0,0,0,54,54,54,54,54,54,54,54,54,54,54,54,54,54
unique,51,1,1,1,54,51,2,0,51,54,6,1,0,0,1,48,50,50,1,1,1,2,1,2,2,1,1,1,2,2,1,2,1,1,1,1,1,1,1,1,1,0,0,0,1,51,1,1,1,0,0,0,0,1,1,51,51,1,51,1,1,2,51,51,1,51,51
top,{'runtime': 1875},{'_target_': 'lmetk.loss.default.MSELoss'},[dev],"{'use_cache_after_loading': True, 'use_cache_a...",{'wandb': {'id': 'unibo+reconstruction+residua...,"{'devices': [1], '_target_': 'lightning.pytorc...",unibo/reconstruction/residual_autoencoder,NaN,{'val': {'paths': {'log_dir': '/home/levtel/pr...,"{'job': {'id': '???', 'num': '???', 'name': '$...",{'_target_': 'lmetk.data.datamodules.base.Base...,{'scaler_features': {'metadata': {'fit_on': 't...,NaN,NaN,debug,"{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...",lmetk.optimizer.default.DefaultOptimizer,Adam,StepLR,lmetk.data.datasources.unibo.loader.UNIBO21Loader,/home/levtel/projects/TabPHM/datasets,UNIBO21,SOC,/home/levtel/projects/TabPHM/datasets/phmd_cache,reconstruction,[features],"[5, 5, 5, 5, 5]","[1, 1, 1, 1, 1]",lmetk.model.autoencoder.wrapper.AutoencoderWra...,"[1, 2, 4, 8, 1]","[16, 32, 64, 64, 64]","[{'in': 2, 'out': 16, 'kernel': 7, 'stride': 2...",lmetk.data.datasets.hydra_concat_dataset.Hydra...,lmetk.data.datasets.sliding_window_batch_datas...,lmetk.callbacks.timer_callback.PipelineTimer,lightning.pytorch.callbacks.RichModelSummary,min,val/loss,lightning.pytorch.callbacks.EarlyStopping,NaN,NaN,NaN,min,/home/levtel/projects/TabPHM/artifacts/debug/r...,val/loss,lightning.pytorch.callbacks.ModelCheckpoint,epoch{epoch:03d}-val_loss{val/loss:.2f},NaN,NaN,NaN,NaN,lightning.pytorch.callbacks.LearningRateMonitor,epoch,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/datasets,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM,/home/levtel/projects/TabPHM,/home/levtel/projects/TabPHM/datasets/cache/UN...,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/artifacts,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPH


--- ⚠️ Columns with NaN Values, 26.42857142857143% has NaN values ---


,num_nans,percentage_nans
ckpt_path,54,100.00
profiler_cfg,54,100.00
subset_range,54,100.00
callbacks.early_stopping.stopping_threshold,54,100.00
callbacks.model_checkpoint.train_time_interval,54,100.00
callbacks.model_checkpoint.save_on_train_epoch_end,54,100.00
callbacks.model_checkpoint.every_n_train_steps,54,100.00
callbacks.early_stopping.check_on_train_epoch_end,54,100.00
callbacks.early_stopping.divergence_threshold,54,100.00
callbacks.model_checkpoint.every_n_epochs,54,100.00


Dropping 13 columns with more than 90% NaNs


### The aim is to find the columns over which the sweep has been done hence are not needed (because they are not part of the sweeped parameters)

In [124]:
def filter_columns(df, filters=None, exclude=False):
    """
    Return columns that contain any of the filter strings, or those that do not contain any of them.
    If filters is None, return all columns.
    If filters is a string, treat as a single filter.
    If filters is a list/tuple, match any of the filters.
    If exclude is True, return columns that do NOT contain any of the filter strings.
    """
    def _matches(col, filters):
        return any(f in col for f in filters)

    # Handle dict input for filters/exclude
    if isinstance(filters, dict):
        filters_ = filters.get("filters", None)
        exclude = filters.get("exclude", False)
    else:
        filters_ = filters

    if filters_ is None:
        return df.columns.tolist()
    if isinstance(filters_, str):
        filters_ = [filters_]

    if exclude:
        return [col for col in df.columns if not _matches(col, filters_)]
    else:
        return [col for col in df.columns if _matches(col, filters_)]




In [125]:

cols_to_drop_1 = filter_columns(df, filters=["callbacks.model_checkpoint.", "paths.", "paths."], exclude=False)


additional_remove_col = [] + cols_to_drop_1
df = df.drop(additional_remove_col, axis=1)
droped_columns.extend(additional_remove_col)

### Some columns that are important might have nan 

In [127]:

# Drop na values if there are
#df = df.dropna(subset=["model.backbone._target_"])
# Reset index
df = df.reset_index(drop=True)

# Drop rows that 'callbacks.early_stopping.monitor' isna
print(
    f"Number of rows with callbacks.early_stopping.monitor = nan is {sum(df['callbacks.early_stopping.monitor'].isna())}"
)



Number of rows with callbacks.early_stopping.monitor = nan is 0


In [142]:
# df["task_definition.model.data_requirements.input_tensors"] elemets are lists of strings
def get_unique_values(df, columns):
    """
    Extract unique values from one or multiple columns in a DataFrame.
    Handles elements that are lists, strings, or other types.
    Args:
        df: pandas DataFrame
        columns: str or list of str, column(s) to extract unique values from
    Returns:
        Set of unique values
    """
    if isinstance(columns, str):
        columns = [columns]
    unique_values = set()
    for col in columns:
        series = df[col].dropna()
        for item in series:
            if isinstance(item, list):
                unique_values.update(item)
            elif isinstance(item, str):
                unique_values.add(item)
            elif pd.api.types.is_scalar(item):
                unique_values.add(item)
            # Optionally handle other types if needed
    return list(unique_values)


In [163]:
import warnings
from collections import defaultdict
import pandas as pd # Assuming you have pandas imported

## Assume df, config_columns, droped_columns, and get_unique_values are defined earlier

# ensure that config_columns exclude the droped_columns is some of dropped colums in in config_columns
config_columns = [col for col in config_columns if col not in droped_columns]


# Identify unique models in DataFrame
unique_models = get_unique_values(df, "model._target_")
# Identify unique datasets in DataFrame
unique_datasets = get_unique_values(df, "datasource.data_name")


collected_results = defaultdict(dict)
collected_results_time = defaultdict(dict)
collected_results_time_run = defaultdict(dict)

collected_aggregated_results = defaultdict(dict)
collected_non_aggregated_results = defaultdict(dict)

# Go over each dataset and model and find the best result
for dataset in unique_datasets:
    for model in unique_models:
        # Get the subset of the DataFrame for the current dataset and model
        subset = df[
            (df["datasource.data_name"] == dataset)
            & (df["model._target_"] == model)
        ]

        if subset.empty:
            print("---------")
            print(f"No results for {model} on {dataset}")
            print("---------")
            continue
            
        # Suppress all warnings temporarily
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            subset["Model"] = model

        def get_metric(df_subset, model_name, dataset_name):
            metric_ = get_unique_values(df=df_subset, columns="callbacks.early_stopping.monitor")
            assert len(metric_) == 1, "There should be only one metric to optimize"
            metric = metric_[0]
            # Print the metric
            print(f"Optimized metric for {model_name} on {dataset_name} is {metric}")
            return metric.split("/")[-1]

        ## <<< CHANGE: Call the function only ONCE and store the result
        metric_name = get_metric(subset, model, dataset)

        ## <<< CHANGE: REUSE the variable to define performance columns
        performance_cols = [f"test/{metric_name}"]

        # Get the unique values for each config column
        unique_colums_values = {}
        for col in config_columns:
            try:
                cleaned_series = subset[col].dropna()
                if cleaned_series.empty:
                    continue

                if isinstance(cleaned_series.iloc[0], list):
                    # Convert lists to tuples to make them hashable, then find unique tuples
                    unique_items = cleaned_series.apply(tuple).unique().tolist()
                    unique_colums_values[col] = unique_items
                else:
                    unique_colums_values[col] = get_unique_values(subset, col)
            except Exception as e:
                print(f"Attention: Column '{col}' had an issue finding unique values: {e}")

        # Print only the config columns that have more than one unique value
        print(f"Hyperparameters that vary for {model} on {dataset}:")
        any_varied = False
        for col, unique in unique_colums_values.items():
            if len(unique) > 1:
                print(f"{col}: {unique}")
                any_varied = True
        if not any_varied:
            print("None.")
        print("---------")

        # Filter to keep only columns that vary for the aggregation step
        varying_columns = {k: v for k, v in unique_colums_values.items() if len(v) > 1}
        
        # Special columns to exclude from aggregation
        special_columns = ["datasource.parameters.data_seed"]
        for col in special_columns:
            if col in varying_columns:
                varying_columns.pop(col)

        aggregation_columns = ["Model"] + list(varying_columns.keys())

        collected_non_aggregated_results[dataset][model] = {
            "df": subset.copy(),
            "aggregation_columns": aggregation_columns,
            "performance_cols": performance_cols,
        }

        # Convert any list columns to tuples for grouping
        for col in aggregation_columns:
            if not subset[col].dropna().empty and isinstance(subset[col].dropna().iloc[0], list):
                subset.loc[:, col] = subset[col].apply(lambda x: tuple(x) if isinstance(x, list) else x)

        # Aggregate the subset
        aggregated = subset.groupby(aggregation_columns).agg(
            {col: ["mean", "std"] for col in performance_cols}
        )
        aggregated = aggregated.reset_index()

        assert len(subset["callbacks.early_stopping.mode"].unique()) == 1, "There should be only one mode"
        
        mode = subset["callbacks.early_stopping.mode"].unique()[0]
        ascending = True if mode == "min" else False

        ## <<< CHANGE: REUSE the variable for sorting
        final_best_ = aggregated.sort_values(
            by=(f"test/{metric_name}", "mean"), ascending=ascending
        ).head(1)

        # Note: final_best_ is already a DataFrame, no need to re-assign
        if mode == "max":
            final_best_ = (final_best_ * 100)
        
        ## <<< CHANGE: REUSE the variable for collecting results
        collected_results[dataset][model] = {
            "mean": final_best_[(f"test/{metric_name}", "mean")].values[0],
            "std": final_best_[(f"test/{metric_name}", "std")].values[0],
        }

        # Get average epoch run time
        collected_results_time[dataset][model] = {
            "mean": subset['AvgTime/train_epoch_mean'].mean(),
            "std": subset['AvgTime/train_epoch_mean'].std(),
        }

        collected_results_time_run[dataset][model] = {
            "mean": subset["_runtime"].mean(),
            "std": subset["_runtime"].std(),
        }

        ## <<< CHANGE: REUSE the variable for the final aggregated results
        collected_aggregated_results[dataset][model] = aggregated.sort_values(
            by=(f"test/{metric_name}", "mean"), ascending=ascending
        )

Optimized metric for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on PRONOSTIA is val/loss
Hyperparameters that vary for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on PRONOSTIA:
model.kernels: [(7, 7, 5, 5, 5), (5, 5, 5, 5, 5)]
model.strides: [(2, 2, 1, 1, 1), (1, 1, 1, 1, 1)]
model.dilations: [(1, 1, 1, 1, 1), (1, 2, 4, 8, 1)]
---------
Optimized metric for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on UNIBO21 is val/loss
Hyperparameters that vary for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on UNIBO21:
optimizer.scheduler.scheduler_params.step_size: [10, 20]
optimizer.parameters.lr: [0.01, 0.001, 0.0005]
model.kernels: [(5, 5, 5, 5, 5), (7, 7, 5, 5, 5)]
model.strides: [(1, 1, 1, 1, 1), (2, 2, 1, 1, 1)]
model.dilations: [(1, 2, 4, 8, 1), (1, 1, 1, 1, 1)]
---------


In [181]:
import warnings
from collections import defaultdict
import pandas as pd

# ===================================================================
#  ASSUMPTIONS (Ensure these are defined before running the script)
# ===================================================================
# df: Your initial pandas DataFrame loaded with all experiment results.
# config_columns: A list of strings with the names of all hyperparameter columns.
# droped_columns: A list of strings with column names to exclude from config_columns.
# get_unique_values(df, columns): A function that returns a list of unique values 
#                                from a specified column in a DataFrame.
#
# Example placeholder for the function:
# def get_unique_values(df, columns):
#     return df[columns].dropna().unique().tolist()
# ===================================================================


# ===================================================================
#  1. HELPER FUNCTIONS (Encapsulated Logic)
# ===================================================================

def get_optimized_metric(df_subset, model_name, dataset_name):
    """Finds and prints the single metric being optimized."""
    metric_series = get_unique_values(df=df_subset, columns="callbacks.early_stopping.monitor")
    if not metric_series:
        raise ValueError(f"No metric found for {model_name} on {dataset_name}. Check 'callbacks.early_stopping.monitor' column.")
    assert len(metric_series) == 1, "There should be only one metric to optimize"
    full_metric_name = metric_series[0]
    print(f"Optimized metric for {model_name} on {dataset_name} is {full_metric_name}")
    return full_metric_name.split("/")[-1]

def get_varying_hyperparameters(df_subset, config_cols, special_cols_to_exclude):
    """
    Identifies which hyperparameters have more than one unique value.
    
    Returns a dictionary of the varying hyperparameters.
    """
    unique_values = {}
    for col in config_cols:
        try:
            cleaned_series = df_subset[col].dropna()
            if cleaned_series.empty:
                continue
            
            # Correctly find unique lists by converting them to tuples first
            if isinstance(cleaned_series.iloc[0], list):
                unique_items = cleaned_series.apply(tuple).unique().tolist()
                unique_values[col] = unique_items
            else:
                unique_values[col] = get_unique_values(df_subset, col)
        except Exception as e:
            print(f"Attention: Could not process column '{col}': {e}")
            
    # Filter for parameters that actually vary
    varying_params = {k: v for k, v in unique_values.items() if len(v) > 1}
    
    # Exclude special columns like data_seed
    for col in special_cols_to_exclude:
        varying_params.pop(col, None) # Use .pop with a default to avoid errors
        
    return varying_params

def aggregate_and_find_best(df_subset, aggregation_cols, performance_col_name):
    """
    Performs the core groupby aggregation and identifies the best performing run.
    
    Returns a tuple containing: (all sorted results, the single best result row).
    """
    subset_copy = df_subset.copy()
    
    # Convert any list-based aggregation columns to tuples so they are hashable
    for col in aggregation_cols:
        if col in subset_copy.columns and not subset_copy[col].dropna().empty and isinstance(subset_copy[col].dropna().iloc[0], list):
            subset_copy.loc[:, col] = subset_copy[col].apply(lambda x: tuple(x) if isinstance(x, list) else x)

    # Perform the aggregation to get mean and std for each hyperparameter group
    aggregated_df = subset_copy.groupby(aggregation_cols).agg(
        {performance_col_name: ["mean", "std"]}
    ).reset_index()

    # Determine sorting direction (min or max)
    mode = subset_copy["callbacks.early_stopping.mode"].unique()[0]
    is_ascending = (mode == "min")

    # Sort all aggregated results to find the best
    sorted_results = aggregated_df.sort_values(by=(performance_col_name, "mean"), ascending=is_ascending)
    
    best_result_row = sorted_results.head(1)
    
    return sorted_results, best_result_row

# ===================================================================
#  2. MAIN SCRIPT LOGIC (High-Level Workflow)
# ===================================================================

# Initial setup
config_columns = [col for col in config_columns if col not in droped_columns]
unique_models = get_unique_values(df, "model._target_")
unique_datasets = get_unique_values(df, "datasource.data_name")
special_columns = ["datasource.parameters.data_seed"]

# Centralized dictionary to store all results
all_results = defaultdict(lambda: defaultdict(dict))

# --- Main processing loop ---
for dataset in unique_datasets:
    for model in unique_models:
        subset = df[(df["datasource.data_name"] == dataset) & (df["model._target_"] == model)].copy()

        if subset.empty:
            print(f"---------\nNo results for {model} on {dataset}\n---------")
            continue
        
        # Add the 'Model' column to be used in aggregation
        # Using .loc on the copy to avoid SettingWithCopyWarning
        subset.loc[:, 'Model'] = model
        
        # --- Step 1: Get key parameters and columns ---
        metric_name = get_optimized_metric(subset, model, dataset)
        performance_col = f"test/{metric_name}"
        varying_hyperparams = get_varying_hyperparameters(subset, config_columns, special_columns)
        aggregation_cols = ["Model"] + list(varying_hyperparams.keys())
        
        # --- Step 2: Print varying hyperparameters for this run ---
        print(f"Hyperparameters that vary for {model} on {dataset}:")
        if not varying_hyperparams:
            print("None.")
        else:
            for col, unique in varying_hyperparams.items():
                print(f"{col}: {unique}")
        print("---------")

        # --- Step 3: Aggregate results and find the best one ---
        sorted_aggregated, best_result = aggregate_and_find_best(subset, aggregation_cols, performance_col)
        
        # --- Step 4: Store all the collected results neatly ---
        mean_val = best_result[(performance_col, "mean")].values[0]
        std_val = best_result[(performance_col, "std")].values[0]
        mode = subset["callbacks.early_stopping.mode"].unique()[0]

        # If maximizing, scale results by 100
        if mode == "max":
            mean_val *= 100
            std_val *= 100
            
        # Store results in the new, more descriptive format
        all_results[dataset][model]['best_performance'] = {
            "metric": metric_name,
            "strategy": mode,
            "mean": mean_val,
            "std": std_val,
        }
        all_results[dataset][model]['avg_epoch_time'] = {"mean": subset['AvgTime/train_epoch_mean'].mean(), "std": subset['AvgTime/train_epoch_mean'].std()}
        all_results[dataset][model]['avg_runtime'] = {"mean": subset["_runtime"].mean(), "std": subset["_runtime"].std()}
        all_results[dataset][model]['sorted_aggregated_results'] = sorted_aggregated
        all_results[dataset][model]['non_aggregated_df'] = subset.copy()

print("\n✅ Analysis complete. Results are stored in the 'all_results' dictionary.")

Optimized metric for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on PRONOSTIA is val/loss
Hyperparameters that vary for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on PRONOSTIA:
model.kernels: [(7, 7, 5, 5, 5), (5, 5, 5, 5, 5)]
model.strides: [(2, 2, 1, 1, 1), (1, 1, 1, 1, 1)]
model.dilations: [(1, 1, 1, 1, 1), (1, 2, 4, 8, 1)]
---------
Optimized metric for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on UNIBO21 is val/loss
Hyperparameters that vary for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on UNIBO21:
optimizer.scheduler.scheduler_params.step_size: [10, 20]
optimizer.parameters.lr: [0.01, 0.001, 0.0005]
model.kernels: [(5, 5, 5, 5, 5), (7, 7, 5, 5, 5)]
model.strides: [(1, 1, 1, 1, 1), (2, 2, 1, 1, 1)]
model.dilations: [(1, 2, 4, 8, 1), (1, 1, 1, 1, 1)]
---------

✅ Analysis complete. Results are stored in the 'all_results' dictionary.


In [185]:
sorted_aggregated

Model  \
                                                        
6   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
0   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
2   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
8   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
9   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
3   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
1   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
7   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
5   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
4   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
11  lmetk.model.autoencoder.wrapper.AutoencoderWra...   
10  lmetk.model.autoencoder.wrapper.AutoencoderWra...   

   optimizer.scheduler.scheduler_params.step_size optimizer.parameters.lr  \
                                                                            
6                                              20                    0.00   
0                                              10                    0.00   
2                                              10                    0.00   
8                                              20                    0.00   
9                                              20                    0.00   
3                                              10                    0.00   
1                                              10                    0.00   
7                                              20                    0.00   
5                                              10                    0.01   
4                                              10                    0.01   
11                                             20                    0.01   
10                                             20                    0.01   

      model.kernels    model.strides  model.dilations test/loss           
                                                           mean      std  
6   (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)      0.00     0.00  
0   (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)      0.00     0.00  
2   (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)      0.00     0.00  
8   (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)      0.00     0.00  
9   (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)      0.00     0.00  
3   (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)      0.00     0.00  
1   (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)      0.00     0.00  
7   (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)      0.00     0.00  
5   (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)      0.02     0.01  
4   (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)      0.10     0.13  
11  (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)      8.16    15.82  
10  (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)  17736.73 34996.48

In [184]:
subset

,_runtime,_step,_timestamp,_wandb,epoch,lr-Adam,train/loss_step,trainer/global_step,loss,seed,tags,test,cache,train,logger,trainer,exp_name,evaluator,hydra_cfg,datamodule,pin_memory,transforms,num_workers,shuffle_val,shuffle_test,shuffle_train,val_batch_size,prefetch_factor,test_batch_size,experiment_group,train_batch_size,use_batch_sampler,model/params/total,model/params/trainable,AvgTime/train_batch_mean,AvgTime/train_batch_std,AvgTime/train_epoch_mean,AvgTime/train_epoch_std,AvgTime/val_batch_mean,AvgTime/val_batch_std,AvgTime/val_epoch_mean,AvgTime/val_epoch_std,plots/test/reconstructions_overview,plots/train/reconstructions_overview,plots/val/reconstructions_overview,test/loss,test/mae,test/mse,test/rmse,train/loss_epoch,train/mae,train/mse,train/rmse,val/loss,val/mae,val/mse,val/rmse,epoch_run_time,optimizer._target_,optimizer.optimizer_id,optimizer.scheduler.scheduler_id,optimizer.scheduler.scheduler_params.gamma,optimizer.scheduler.scheduler_params.step_size,optimizer.parameters.lr,optimizer.parameters.weight_decay,datasource._target_,datasource.data_dir,datasource.data_name,datasource.task_mode,task_definition.stride,task_definition.seq_len,task_definition.pred_len,task_definition.label_len,task_definition.task_type,task_definition.requires_training,task_definition.model.data_requirements.input_tensors,model.kernels,model.strides,model._target_,model.dilations,model.latent_dim,model.dropout_prob,model.input_seq_len,model.input_channels,model.output_channels,dataset._target_,dataset._recursive_,dataset.dataset_cfg.stride,dataset.dataset_cfg.seq_len,dataset.dataset_cfg._target_,dataset.dataset_cfg.pred_len,dataset.dataset_cfg.label_len,callbacks.model_timer._target_,callbacks.model_summary._target_,callbacks.model_summary.max_depth,callbacks.early_stopping.mode,callbacks.early_stopping.strict,callbacks.early_stopping.monitor,callbacks.early_stopping.verbose,callbacks.early_stopping._target_,callbacks.early_stopping.patience,callbacks.early_stopping.min_delta,callbacks.early_stopping.check_finite,callbacks.learning_rate_monitor._target_,callbacks.learning_rate_monitor.logging_interval,Model
0,107.00,127,1759357555.33,{'runtime': 107},0,0.01,0.00,6349,{'_target_': 'lmetk.loss.default.MSELoss'},42,[dev],True,"{'use_cache_after_loading': True, 'use_cache_a...",True,{'wandb': {'id': 'unibo+reconstruction+residua...,"{'devices': [1], '_target_': 'lightning.pytorc...",unibo/reconstruction/residual_autoencoder,{'val': {'paths': {'log_dir': '/home/levtel/pr...,"{'job': {'id': '???', 'num': '???', 'name': '$...",{'_target_': 'lmetk.data.datamodules.base.Base...,True,{'scaler_features': {'metadata': {'fit_on': 't...,4,False,False,True,16,10,16,debug,16,True,1167436,1167436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,inf,lmetk.optimizer.default.DefaultOptimizer,Adam,StepLR,0.50,10,0.01,0,lmetk.data.datasources.unibo.loader.UNIBO21Loader,/home/levtel/projects/TabPHM/datasets,UNIBO21,SOC,1,64,0,0,reconstruction,True,[features],"[5, 5, 5, 5, 5]","[1, 1, 1, 1, 1]",lmetk.model.autoencoder.wrapper.AutoencoderWra...,"[1, 2, 4, 8, 1]",128,0.20,64,3,"[16, 32, 64, 64, 64]",lmetk.data.datasets.hydra_concat_dataset.Hydra...,False,1,64,lmetk.data.datasets.sliding_window_batch_datas...,0,0,lmetk.callbacks.timer_callback.PipelineTimer,lightning.pytorch.callbacks.RichModelSummary,-1,min,True,val/loss,True,lightning.pytorch.callbacks.EarlyStopping,25,0,True,lightning.pytorch.callbacks.LearningRateMonitor,epoch,lmetk.model.autoencoder.wrapper.AutoencoderWra...
1,107.00,121,1759357555.34,{'runtime': 107},0,0.01,0.00,6049,{'_target_': 'lmetk.loss.default.MSELoss'},42,[dev],True,"{'use_cache_after_loading': True, 'use_cache_a...",True,{'wandb': {'id': 'unibo+reconstruction+residua...,"{'devices': [1], '_target_': 'lightning.pytorc...",unibo/reconstruction/residual_autoencoder,{'val': {'paths': {'log_dir': '/home/levtel/pr...,"{'job': {'id': '???', 'num': '???', 'name': '$...",{'_target_': 'lmetk.data.data

In [180]:
all_results['UNIBO21']['lmetk.model.autoencoder.wrapper.AutoencoderWrapper']['best_performance']

{'metric': 'loss',
 'strategy': 'min',
 'mean': np.float64(1.9044728638561992e-06),
 'std': np.float64(2.844459746217841e-06)}

In [164]:
collected_aggregated_results['UNIBO21']['lmetk.model.autoencoder.wrapper.AutoencoderWrapper']

Model  \
                                                        
6   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
0   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
2   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
8   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
9   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
3   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
1   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
7   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
5   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
4   lmetk.model.autoencoder.wrapper.AutoencoderWra...   
11  lmetk.model.autoencoder.wrapper.AutoencoderWra...   
10  lmetk.model.autoencoder.wrapper.AutoencoderWra...   

   optimizer.scheduler.scheduler_params.step_size optimizer.parameters.lr  \
                                                                            
6                                              20                    0.00   
0                                              10                    0.00   
2                                              10                    0.00   
8                                              20                    0.00   
9                                              20                    0.00   
3                                              10                    0.00   
1                                              10                    0.00   
7                                              20                    0.00   
5                                              10                    0.01   
4                                              10                    0.01   
11                                             20                    0.01   
10                                             20                    0.01   

      model.kernels    model.strides  model.dilations test/loss           
                                                           mean      std  
6   (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)      0.00     0.00  
0   (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)      0.00     0.00  
2   (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)      0.00     0.00  
8   (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)      0.00     0.00  
9   (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)      0.00     0.00  
3   (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)      0.00     0.00  
1   (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)      0.00     0.00  
7   (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)      0.00     0.00  
5   (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)      0.02     0.01  
4   (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)      0.10     0.13  
11  (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)      8.16    15.82  
10  (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)  17736.73 34996.48

### Runtime per epoch

In [36]:
collected_results_time
# Convert nested dictionary to DataFrame
nested_dict = dict(collected_results_time)
result_dict = pd.DataFrame.from_dict(
    {
        (i, j): nested_dict[i][j]
        for i in nested_dict
        for j in nested_dict[i].keys()
    },
    orient="index",
)


result_dict = result_dict.round(2)
result_dict["performance"] = result_dict.apply(
    lambda x: f"{x['mean']} ± {x['std']}", axis=1
)
result_dict = result_dict.drop(["mean", "std"], axis=1)

# Reset multiindex
result_dict = result_dict.reset_index()
# rename columns
result_dict.columns = ["Dataset", "Model", "Average Time per Epoch"]

In [38]:
result_dict

,Dataset,Model,Average Time per Epoch
0,ogbg-molhiv,SANN,15.75 ± 2.78


In [39]:
result_dict.pivot_table(
    index="Model", columns="Dataset", values="Average Time per Epoch", aggfunc="first"
)[['ogbg-molhiv']] #[['MUTAG', 'NCI1','NCI109','PROTEINS','ZINC']]

Dataset,ogbg-molhiv
Model,
SANN,15.75 ± 2.78


### Runtime

In [40]:
collected_results_time_run
# Convert nested dictionary to DataFrame
nested_dict = dict(collected_results_time_run)
result_dict = pd.DataFrame.from_dict(
    {
        (i, j): nested_dict[i][j]
        for i in nested_dict
        for j in nested_dict[i].keys()
    },
    orient="index",
)


result_dict = result_dict.round(2)
result_dict["performance"] = result_dict.apply(
    lambda x: f"{x['mean']} ± {x['std']}", axis=1
)
result_dict = result_dict.drop(["mean", "std"], axis=1)

# Reset multiindex
result_dict = result_dict.reset_index()
# rename columns
result_dict.columns = ["Dataset", "Model", "Average Training Time"]


In [43]:
result_dict.pivot_table(
    index="Model", columns="Dataset", values="Average Training Time", aggfunc="first"
)[['ogbg-molhiv']] #[['MUTAG', 'NCI1','NCI109','PROTEINS','ZINC']]

Dataset,ogbg-molhiv
Model,
SANN,1378.66 ± 306.25
